# Modelado Estocástico
## Clase 5 - Regresión Múltiple - Ejemplo de Multicolinealidad y Tests de Heteroscedasticidad

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.stats.api as sms
import seaborn as sns
import statsmodels.stats.diagnostic as diag
from statsmodels.stats.diagnostic import het_white, het_breuschpagan

## Ejemplo de Multicolinealidad

La multicolinealidad no sesga los estimadores de Mínimos Cuadrados Ordinarios (MCO), pero infla su varianza. Matemáticamente, la matriz de varianzas y covarianzas de los estimadores $\hat{\beta}$ está dada por:$$\text{Var}(\hat{\beta}) = \sigma^2 (\mathbf{X}^T\mathbf{X})^{-1}$$

Si existe una combinación lineal exacta entre las columnas de  $\mathbf{X}$ , el determinante de  $\mathbf{X}^T\mathbf{X}$ es cero, impidiendo su inversión. Si es casi perfecta, el determinante es infinitesimalmente pequeño, haciendo que los elementos de la diagonal de  $(\mathbf{X}^T\mathbf{X})^{-1}$  exploten.

In [ ]:
dfmulti = pd.read_excel('CEO_ejemplo_multicolinealidad.xlsx', sheet_name='Hoja1', usecols='A:C')
dfmulti.head(20)

,Gan,Gan_10,Comp
0,357.0,35,0.7
1,48.0,4,0.7
2,932.0,93,0.8
3,366.0,36,0.7
4,83.0,8,0.8
5,22.0,2,0.0
6,67.0,6,0.0
7,413.0,41,0.6
8,496.0,49,0.3
9,458.0,45,0.5


### p-values de betas Gan y Gan_10 grandes, indicando que no rechazamos $H_0: b=0$


In [ ]:
X_mult = sm.add_constant(dfmulti.drop(columns='Comp'))
y = dfmulti["Comp"]
regre_multico = sm.OLS(y, X_mult).fit()
print(regre_multico.summary())

                            OLS Regression Results                            
Dep. Variable:                   Comp   R-squared:                       0.436
Model:                            OLS   Adj. R-squared:                  0.419
Method:                 Least Squares   F-statistic:                     25.92
Date:                Sat, 08 Aug 2026   Prob (F-statistic):           4.59e-09
Time:                        12:54:36   Log-Likelihood:                -73.546
No. Observations:                  70   AIC:                             153.1
Df Residuals:                      67   BIC:                             159.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6558      0.166      3.941      0.0

En estos casos, un test F sobre las variables que resultan ser no significativas es especialmente útil, ya que si rechazamos la hipótesis nula de que simultáneamente sus betas que las multiplican son cero, entonces podemos concluir que estamos en presencia de mulktiucolinealidad. Y si no rechazamos la hipótesis nula, entonces las variables explicativas involucradas en este test simplemente no explican a la variable dependiente.

In [ ]:
print(regre_multico.f_test("Gan = 0, Gan_10 = 0"))

<F test: F=25.92225542717165, p=4.5888632880021224e-09, df_denom=67, df_num=2>


Vemos en el test F de arriba que se rechaza la hipótesis nula de que ambos betas son simultáneamente iguales a cero. Procedemos eliminando de a una a la vez las variables no significativas (si se tratara de más de 2 variables).

In [ ]:
X = sm.add_constant(dfmulti["Gan"])
regre_normal = sm.OLS(y, X).fit()
print(regre_normal.summary())

                            OLS Regression Results                            
Dep. Variable:                   Comp   R-squared:                       0.434
Model:                            OLS   Adj. R-squared:                  0.426
Method:                 Least Squares   F-statistic:                     52.24
Date:                Sat, 01 Aug 2026   Prob (F-statistic):           5.50e-10
Time:                        11:15:55   Log-Likelihood:                -73.655
No. Observations:                  70   AIC:                             151.3
Df Residuals:                      68   BIC:                             155.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.6000      0.112      5.342      0.0

## Ejemplos de tests de Heteroscedasticidad

Retomamos el dataset de los precios de viviendas para realizar tests para detectar heteroscedasticidad.

In [ ]:
df_casa = pd.read_excel('Ejemplo_Casa.xls', sheet_name='HPRICE', usecols='A:L')
X = df_casa.drop(columns='PRECIO')
X = sm.add_constant(X)
y = df_casa["PRECIO"]
regre_casa = sm.OLS(y, X).fit()

## Test de Heteroscedasticidad de White

In [ ]:
white_test = sms.het_white(regre_casa.resid, regre_casa.model.exog)
print(f"LM p-value (el estadístico): {white_test[0]}      p-value: {white_test[1]}")

LM p-value (el estadístico): 168.27932387737198      p-value: 7.144729800456718e-10


Alternativamente, con `statsmodels.stats.diagnostic`.

https://www.statsmodels.org/devel/generated/statsmodels.stats.diagnostic.het_white.html

In [ ]:
lm_stat, lm_pval, f_stat, f_pval = het_white(regre_casa.resid, X)
print(f"Test de White:  Estadístico LM = {lm_stat} | p-valor = {lm_pval}")

Test de White:  Estadístico LM = 168.27932387737198 | p-valor = 7.144729800456718e-10


Corrección de la heterocedasticidad (corrige la segunda columna)

Resultados con errores estándar robustos (tipo White / HC1)

https://www.statsmodels.org/devel/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html


In [ ]:
regre_casa_robusto = regre_casa.get_robustcov_results(cov_type='HC1')
print(regre_casa_robusto.summary())

                            OLS Regression Results                            
Dep. Variable:                 PRECIO   R-squared:                       0.673
Model:                            OLS   Adj. R-squared:                  0.666
Method:                 Least Squares   F-statistic:                     87.32
Date:                Sat, 01 Aug 2026   Prob (F-statistic):          1.05e-111
Time:                        11:15:55   Log-Likelihood:                -6034.1
No. Observations:                 546   AIC:                         1.209e+04
Df Residuals:                     534   BIC:                         1.214e+04
Df Model:                          11                                         
Covariance Type:                  HC1                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -4038.3504   3182.329     -1.269      0.2

## Test de Breusch-Pagan

https://www.statsmodels.org/devel/generated/statsmodels.stats.diagnostic.het_breuschpagan.html

In [ ]:
bp_stat, bp_pval, _, _ = het_breuschpagan(regre_casa.resid, X)
print(f"Test de Breusch-Pagan: Estadístico LM = {bp_stat} | p-valor = {bp_pval}")

Test de Breusch-Pagan: Estadístico LM = 61.9525822872857 | p-valor = 4.01376194878812e-09


## Test de Goldfeld y Quandt

https://www.statsmodels.org/dev/generated/statsmodels.stats.diagnostic.het_goldfeldquandt.html

In [ ]:
import statsmodels.stats.diagnostic as diag

X = df_casa.drop(columns='PRECIO')
X = sm.add_constant(X)
y = df_casa["PRECIO"]
regre_casa = sm.OLS(y, X).fit()

n = len(y)
c = int(0.20 * n)

idx_lote = X.columns.get_loc('LOTE')

gq_stat, gq_pval, gq_dir = diag.het_goldfeldquandt(
    y=regre_casa.model.endog,
    x=regre_casa.model.exog,
    idx=idx_lote,
    drop= c,
    alternative='increasing'
)

print(f"Estadístico F de GQ: {gq_stat:.4f}")
print(f"p-valor de GQ:       {gq_pval:.6e}")
print(f"Dirección del Test:  {gq_dir}")

Estadístico F de GQ: 2.3853
p-valor de GQ:       5.672506e-09
Dirección del Test:  increasing
